# Avro Basics — 07: Kafka Simulation


The most important Avro use case: Kafka message serialization.
Kafka uses Avro with a Schema Registry — schema ID embedded in message bytes.

Topics: Confluent wire format (magic byte + schema_id + payload),
serialize/deserialize pattern, schema registry simulation, batch processing.


In [1]:
import os, time, pathlib, shutil, random, datetime, json as pyjson
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/avro_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('avro-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Avro JAR: spark-avro_2.13-4.0.2")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 13:05:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/usr/local/lib/python3.10/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

## Step 1 — Confluent Wire Format

Magic byte 0x00 + 4-byte schema_id + Avro payload.

In [ ]:

import struct, json as pyjson

SCHEMA_ID_V1 = 1
SCHEMA_ID_V2 = 2

EVENT_SCHEMA = pyjson.dumps({"type":"record","name":"ClickEvent","fields":[
    {"name":"event_id","type":"long"},
    {"name":"user_id", "type":"long"},
    {"name":"page",    "type":"string"},
    {"name":"revenue", "type":"double"},
]})

def make_kafka_message(schema_id, data):
    """Simulate Confluent Kafka Avro message: [0x00][4-byte schema_id][json_payload]"""
    header  = bytes([0x00]) + struct.pack(">I", schema_id)
    payload = pyjson.dumps(data).encode("utf-8")
    return header + payload

def parse_kafka_message(msg):
    """Deserialize Confluent Kafka Avro message."""
    assert msg[0] == 0, "Not a Confluent Avro message"
    schema_id = struct.unpack(">I", msg[1:5])[0]
    payload   = pyjson.loads(msg[5:].decode("utf-8"))
    return schema_id, payload

# Simulate 5 messages
messages = [
    make_kafka_message(SCHEMA_ID_V1, {"event_id":i,"user_id":100+i,
                                      "page":"/home","revenue":0.0 if i%2 else 99.99})
    for i in range(1,6)
]
print(f"Produced {len(messages)} Kafka messages")
print(f"Message size: {len(messages[0])} bytes (5 header + payload)")
print()
print("Consuming messages:")
for msg in messages:
    sid, data = parse_kafka_message(msg)
    print(f"  schema_id={sid} event_id={data['event_id']} revenue={data['revenue']}")


## Step 2 — Schema Registry Simulation



In [ ]:

import pathlib

REGISTRY = {}  # schema_id → schema JSON

def register_schema(schema_json):
    global REGISTRY
    sid = len(REGISTRY) + 1
    REGISTRY[sid] = schema_json
    return sid

def get_schema(schema_id):
    return REGISTRY.get(schema_id)

# Register two schema versions
sid_v1 = register_schema(pyjson.dumps({"type":"record","name":"Event","fields":[
    {"name":"event_id","type":"long"},
    {"name":"user_id", "type":"long"},
    {"name":"page",    "type":"string"},
    {"name":"revenue", "type":"double"},
]}))

sid_v2 = register_schema(pyjson.dumps({"type":"record","name":"Event","fields":[
    {"name":"event_id","type":"long"},
    {"name":"user_id", "type":"long"},
    {"name":"page",    "type":"string"},
    {"name":"revenue", "type":"double"},
    {"name":"device",  "type":"string", "default":"unknown"},
    {"name":"country", "type":["null","string"], "default":None},
]}))

print(f"Registered schemas: v1=id{sid_v1}, v2=id{sid_v2}")
print()

# Mixed messages: old producer uses v1, new producer uses v2
mixed = [make_kafka_message(sid_v1, {"event_id":i,"user_id":i,"page":"/a","revenue":0.0}) for i in range(3)]
mixed += [make_kafka_message(sid_v2, {"event_id":i+3,"user_id":i+3,"page":"/b","revenue":9.99,"device":"mobile","country":"US"}) for i in range(3)]

print("Processing mixed schema messages:")
for msg in mixed:
    sid, data = parse_kafka_message(msg)
    schema = get_schema(sid)
    fields = [f["name"] for f in pyjson.loads(schema)["fields"]]
    print(f"  schema_id={sid} fields={len(fields)} event_id={data['event_id']}")


## Step 3 — Batch Processing Kafka Avro in Spark



In [ ]:

# Simulate: Kafka messages written to files, processed in Spark
all_messages = []
for i in range(1000):
    sid = sid_v1 if i < 700 else sid_v2  # 70% v1, 30% v2
    data = {"event_id": i, "user_id": i%500, "page": "/page", "revenue": float(i%100)}
    if sid == sid_v2:
        data["device"]  = "mobile" if i%2==0 else "desktop"
        data["country"] = "US" if i%3==0 else "UK"
    all_messages.append((make_kafka_message(sid, data),))

msg_df = spark.createDataFrame(all_messages, ["raw_bytes"])

# Deserialize in Spark using Python UDF
import pyspark.sql.functions as F2
from pyspark.sql.types import MapType, StringType

@F.udf(returnType=StringType())
def extract_payload(msg_bytes):
    if msg_bytes and len(msg_bytes) > 5:
        return msg_bytes[5:].decode("utf-8","replace")
    return None

@F.udf(returnType=LongType())
def extract_schema_id(msg_bytes):
    if msg_bytes and len(msg_bytes) >= 5:
        return int(struct.unpack(">I", bytes(msg_bytes[1:5]))[0])
    return None

parsed = msg_df \
    .withColumn("schema_id", extract_schema_id(F.col("raw_bytes"))) \
    .withColumn("payload",   extract_payload(F.col("raw_bytes")))

print(f"Parsed {parsed.count()} messages:")
parsed.groupBy("schema_id").count().show()


## Step 4 — Normalize to Parquet Landing Zone



In [ ]:

import json as pyjson

# Parse payload JSON and normalize mixed schemas
@F.udf(returnType=LongType())
def get_event_id(payload):
    try: return pyjson.loads(payload)["event_id"]
    except: return None

@F.udf(returnType=DoubleType())
def get_revenue(payload):
    try: return float(pyjson.loads(payload).get("revenue",0))
    except: return None

@F.udf(returnType=StringType())
def get_device(payload):
    try: return pyjson.loads(payload).get("device","unknown")
    except: return "unknown"

from pyspark.sql.types import *
parsed_full = msg_df \
    .withColumn("schema_id", extract_schema_id("raw_bytes")) \
    .withColumn("payload",   extract_payload("raw_bytes")) \
    .withColumn("event_id",  get_event_id("payload")) \
    .withColumn("revenue",   get_revenue("payload")) \
    .withColumn("device",    get_device("payload")) \
    .drop("raw_bytes","payload")

print("Normalized DataFrame:")
parsed_full.show(5)

parsed_full.write.mode("overwrite").parquet(f"{DATA_DIR}/kafka_landing")
print(f"\nLanding zone: {spark.read.parquet(f'{DATA_DIR}/kafka_landing').count():,} rows in Parquet")


## Summary



In [ ]:

print("""
Kafka Avro pipeline:
  Producer → Schema Registry (register schema, get ID)
           → Serialize: [0x00][4-byte schema_id][avro_payload]
           → Publish to Kafka topic

  Consumer → Read message bytes
           → Extract schema_id (bytes 1-4)
           → Lookup schema from registry
           → Deserialize payload

In Spark (Kafka source):
  spark.readStream
       .format("kafka")
       .option("kafka.bootstrap.servers","host:9092")
       .option("subscribe","events")
       .load()
  → value column contains raw bytes
  → use from_avro(col("value"), schema) to deserialize

Landing zone pattern:
  Kafka → Avro bytes → Spark deserialize → normalize → Parquet
""")
